## Etapa 1

In [56]:
from pyspark.sql import SparkSession
from pyspark import SparkContext, SQLContext
from pyspark.sql.functions import  expr,when, rand
from pyspark.sql import functions as F


spark = SparkSession.builder.master("local").appName("Exercicio Intro").getOrCreate()


names = "nomes_aleatorios.txt"

# Lendo o arquivo nomes_aleatorios.txt com Spark
df_nomes = spark.read.csv(names, header=False, inferSchema=True)

# Exibindo as primeiras 5 linhas do DataFrame
df_nomes.show(5)


+----------------+
|             _c0|
+----------------+
|  Frances Bennet|
|   Jamie Russell|
|  Edward Kistler|
|   Sheila Maurer|
|Donald Golightly|
+----------------+
only showing top 5 rows



## Etapa 2

In [59]:

df_nomes.printSchema()

df_nomes = df_nomes.withColumnRenamed("_c0","Nomes")

df_nomes.show(10)


root
 |-- _c0: string (nullable = true)

+-----------------+
|            Nomes|
+-----------------+
|   Frances Bennet|
|    Jamie Russell|
|   Edward Kistler|
|    Sheila Maurer|
| Donald Golightly|
|       David Gray|
|      Joy Bennett|
|      Paul Kriese|
|Berniece Ornellas|
|    Brian Farrell|
+-----------------+
only showing top 10 rows



## Etapa 3

In [60]:
df_nomes = df_nomes.withColumn(
    "Escolaridade",
    when(rand() < 0.33, "Fundamental")
    .when(rand() < 0.66, "Medio")
    .otherwise("Superior")
)

df_nomes.show(10)

+-----------------+------------+
|            Nomes|Escolaridade|
+-----------------+------------+
|   Frances Bennet|       Medio|
|    Jamie Russell| Fundamental|
|   Edward Kistler|    Superior|
|    Sheila Maurer| Fundamental|
| Donald Golightly|       Medio|
|       David Gray|    Superior|
|      Joy Bennett| Fundamental|
|      Paul Kriese|       Medio|
|Berniece Ornellas| Fundamental|
|    Brian Farrell| Fundamental|
+-----------------+------------+
only showing top 10 rows



## Etapa 4


In [61]:
paises = [ "Argentina", "Bolívia", "Brasil", "Chile", "Colômbia", 
"Equador", "Guiana", "Paraguai", "Peru", "Suriname", 
"Uruguai", "Venezuela", "Guiana Francesa"]

df_nomes = df_nomes.withColumn(
    "Pais",
    expr(f"array({', '.join(map(lambda x: f'\"{x}\"', paises))})[int(rand() * {len(paises)})]"
))

df_nomes.show(10)

+-----------------+------------+---------+
|            Nomes|Escolaridade|     Pais|
+-----------------+------------+---------+
|   Frances Bennet|       Medio| Suriname|
|    Jamie Russell| Fundamental|    Chile|
|   Edward Kistler|    Superior|  Equador|
|    Sheila Maurer| Fundamental|  Uruguai|
| Donald Golightly|       Medio|     Peru|
|       David Gray|    Superior|Venezuela|
|      Joy Bennett| Fundamental|   Brasil|
|      Paul Kriese|       Medio|Venezuela|
|Berniece Ornellas| Fundamental|Argentina|
|    Brian Farrell| Fundamental|   Brasil|
+-----------------+------------+---------+
only showing top 10 rows



## Etapa 5

In [62]:

df_nomes = df_nomes.withColumn(
    "Ano_Nascimento",
    expr("1945 + int(rand() * 66)")  
)


df_nomes.show(10)

+-----------------+------------+---------+--------------+
|            Nomes|Escolaridade|     Pais|Ano_Nascimento|
+-----------------+------------+---------+--------------+
|   Frances Bennet|       Medio| Suriname|          1997|
|    Jamie Russell| Fundamental|    Chile|          1958|
|   Edward Kistler|    Superior|  Equador|          1954|
|    Sheila Maurer| Fundamental|  Uruguai|          2003|
| Donald Golightly|       Medio|     Peru|          1982|
|       David Gray|    Superior|Venezuela|          1975|
|      Joy Bennett| Fundamental|   Brasil|          1992|
|      Paul Kriese|       Medio|Venezuela|          1988|
|Berniece Ornellas| Fundamental|Argentina|          1967|
|    Brian Farrell| Fundamental|   Brasil|          1991|
+-----------------+------------+---------+--------------+
only showing top 10 rows



# Etapa 6

In [63]:

df_select = df_nomes.filter(df_nomes["Ano_Nascimento"] >= 2001)

df_select.show(10)

+----------------+------------+---------+--------------+
|           Nomes|Escolaridade|     Pais|Ano_Nascimento|
+----------------+------------+---------+--------------+
|   Sheila Maurer| Fundamental|  Uruguai|          2003|
| Helen Blackwell| Fundamental|Argentina|          2006|
|     Frank Wiley|       Medio|  Bolívia|          2004|
|  Gabriel Colyer| Fundamental|     Peru|          2003|
|  Wilfredo Grant|       Medio|Argentina|          2008|
|  Katrina Graham| Fundamental|   Brasil|          2006|
|   Ricky Gilbert| Fundamental|Venezuela|          2003|
|Flossie Anderson|    Superior|  Uruguai|          2006|
|     Lisa Baxley|       Medio|Venezuela|          2009|
|  Ashley Trosper|       Medio| Suriname|          2005|
+----------------+------------+---------+--------------+
only showing top 10 rows



# Etapa 7

In [64]:
df_nomes.createOrReplaceTempView("pessoas")

df_select = spark.sql("SELECT * FROM pessoas WHERE  Ano_Nascimento >= 2001").show()



+--------------------+------------+---------------+--------------+
|               Nomes|Escolaridade|           Pais|Ano_Nascimento|
+--------------------+------------+---------------+--------------+
|       Sheila Maurer| Fundamental|        Uruguai|          2003|
|     Helen Blackwell| Fundamental|      Argentina|          2006|
|         Frank Wiley|       Medio|        Bolívia|          2004|
|      Gabriel Colyer| Fundamental|           Peru|          2003|
|      Wilfredo Grant|       Medio|      Argentina|          2008|
|      Katrina Graham| Fundamental|         Brasil|          2006|
|       Ricky Gilbert| Fundamental|      Venezuela|          2003|
|    Flossie Anderson|    Superior|        Uruguai|          2006|
|         Lisa Baxley|       Medio|      Venezuela|          2009|
|      Ashley Trosper|       Medio|       Suriname|          2005|
|Christopher Williams|    Superior|Guiana Francesa|          2002|
|       Jenna Johnson| Fundamental|           Peru|          2

# Etapa 8

In [ ]:

df_millennials = df_nomes.filter(F.col("Ano_Nascimento").between(1980, 1994))

df_millennials.show(10)

+----------------+------------+---------+--------------+
|           Nomes|Escolaridade|     Pais|Ano_Nascimento|
+----------------+------------+---------+--------------+
|Donald Golightly|       Medio|     Peru|          1982|
|     Joy Bennett| Fundamental|   Brasil|          1992|
|     Paul Kriese|       Medio|Venezuela|          1988|
|   Brian Farrell| Fundamental|   Brasil|          1991|
|  Kara Mcelwaine| Fundamental|   Brasil|          1990|
|    Ernest Hulet|    Superior|Venezuela|          1984|
|    Charles Hill| Fundamental|  Uruguai|          1985|
|         Lois Ly| Fundamental|  Bolívia|          1985|
| Kenneth Rayburn| Fundamental|    Chile|          1989|
|  Mary Dillahunt| Fundamental|  Bolívia|          1985|
+----------------+------------+---------+--------------+
only showing top 10 rows



# Etapa 9

In [67]:
df_nomes.createOrReplaceTempView("pessoas")

df_select = spark.sql("SELECT * FROM millennials WHERE  Ano_Nascimento BETWEEN 1980 and 1994").show()


+-----------------+------------+---------------+--------------+
|            Nomes|Escolaridade|           Pais|Ano_Nascimento|
+-----------------+------------+---------------+--------------+
| Donald Golightly|       Medio|           Peru|          1982|
|      Joy Bennett| Fundamental|         Brasil|          1992|
|      Paul Kriese|       Medio|      Venezuela|          1988|
|    Brian Farrell| Fundamental|         Brasil|          1991|
|   Kara Mcelwaine| Fundamental|         Brasil|          1990|
|     Ernest Hulet|    Superior|      Venezuela|          1984|
|     Charles Hill| Fundamental|        Uruguai|          1985|
|          Lois Ly| Fundamental|        Bolívia|          1985|
|  Kenneth Rayburn| Fundamental|          Chile|          1989|
|   Mary Dillahunt| Fundamental|        Bolívia|          1985|
|       Ned Tester|       Medio|         Brasil|          1991|
|      Hugo Dayton| Fundamental|      Argentina|          1988|
|Marguerite Rankin|       Medio|       S

# Etapa 10

In [ ]:
df_nomes.createOrReplaceTempView("pessoas")

df_select = spark.sql(
                      """
    SELECT 
        Pais,
        CASE
            WHEN Ano_Nascimento BETWEEN 1944 AND 1964 THEN 'Baby Boomers'
            WHEN Ano_Nascimento BETWEEN 1965 AND 1979 THEN 'Geração X'
            WHEN Ano_Nascimento BETWEEN 1980 AND 1994 THEN 'Millennials (Geração Y)'
            WHEN Ano_Nascimento BETWEEN 1995 AND 2015 THEN 'Geração Z'
        END AS Geracao,
        COUNT(*) AS Quantidade
    FROM pessoas
    GROUP BY Pais, Geracao
    ORDER BY Pais, Geracao, Quantidade
"""
).show()

ParseException: 
[PARSE_SYNTAX_ERROR] Syntax error at or near '('.(line 11, pos 13)

== SQL ==

    SELECT 
        Pais,
        CASE
            WHEN Ano_Nascimento BETWEEN 1944 AND 1964 THEN 'Baby Boomers'
            WHEN Ano_Nascimento BETWEEN 1965 AND 1979 THEN 'Geração X'
            WHEN Ano_Nascimento BETWEEN 1980 AND 1994 THEN 'Millennials (Geração Y)'
            WHEN Ano_Nascimento BETWEEN 1995 AND 2015 THEN 'Geração Z'
            ELSE 'Outra Geração'
        END
        COUNT(*) AS Quantidade
-------------^^^
    FROM pessoas
    GROUP BY Pais, Geracao
    ORDER BY Pais, Geracao, Quantidade
